# Text Classification Task

Is module mein, hum **[AG_NEWS](http://www.di.unipi.it/~gulli/AG_corpus_of_news_articles.html)** dataset par based ek simple text classification task se shuru karenge: hum news headlines ko chaar categories mein classify karenge: World, Sports, Business aur Sci/Tech.

## Dataset

Dataset load karne ke liye, hum **[TensorFlow Datasets](https://www.tensorflow.org/datasets)** API use karenge.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds

# In this tutorial, we will be training a lot of models. In order to use GPU memory cautiously,
# we will set tensorflow option to grow GPU memory allocation when required.
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

dataset = tfds.load('ag_news_subset')

Ab hum `dataset['train']` aur `dataset['test']` se dataset ke training aur test parts tak pahunch sakte hain:

In [ ]:
ds_train = dataset['train']
ds_test = dataset['test']

print(f"Length of train dataset = {len(ds_train)}")
print(f"Length of test dataset = {len(ds_test)}")

Chaliye apne dataset se pehli 10 new headlines print karte hain:

In [ ]:
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

for i,x in zip(range(5),ds_train):
    print(f"{x['label']} ({classes[x['label']]}) -> {x['title']} {x['description']}")

## Text Vectorization

Ab hamein text ko **numbers** mein badalna hoga, jinhe tensors ke roop mein represent kiya ja sake. Agar hamein word-level representation chahiye, toh hamein do cheezein karni hongi:

* Ek **tokenizer** use karke text ko **tokens** mein split karein.
* Un tokens ki ek **vocabulary** banayein.

### Vocabulary size ko limit karna

AG News dataset ke example mein, vocabulary size kaafi bada hai, 100k se zyada words. Aam taur par, hamein un words ki zaroorat nahi hoti jo text mein bahut kam baar aate hain — woh sirf kuch sentences mein hi honge, aur model unse kuch seekh nahi paayega. Isliye, vocabulary size ko chhota rakhne ke liye ise limit karna samajhdaari hai. Yeh vectorizer constructor mein ek argument pass karke kiya ja sakta hai:

In dono steps ko **TextVectorization** layer se handle kiya ja sakta hai. Chaliye vectorizer object instantiate karte hain, aur phir `adapt` method call karte hain taaki saare text se guzar kar vocabulary ban sake:

In [ ]:
vocab_size = 50000
vectorizer = keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size)
vectorizer.adapt(ds_train.take(500).map(lambda x: x['title']+' '+x['description']))

> **Note** Hum vocabulary banane ke liye poore dataset ka sirf ek chhota hissa use kar rahe hain. Aisa hum execution time tez karne aur aapko intezaar na karwane ke liye kar rahe hain. Lekin, hum yeh risk le rahe hain ki poore dataset ke kuch words vocabulary mein shamil nahi honge aur training ke dauran ignore ho jayenge. Isliye, poori vocabulary size use karna aur `adapt` ke dauran poore dataset se guzarna final accuracy badha sakta hai, lekin bahut zyada nahi.

Ab hum actual vocabulary tak pahunch sakte hain:

In [ ]:
vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)
print(vocab[:10])
print(f"Length of vocabulary: {vocab_size}")

Vectorizer use karke, hum kisi bhi text ko aasaani se numbers ke set mein encode kar sakte hain:

In [ ]:
vectorizer('I love to play with my words')

## Bag-of-Words text representation

Kyunki words meaning dikhate hain, kabhi-kabhi hum sirf individual words ko dekhkar text ka meaning pata laga sakte hain, chahe woh sentence mein kisi bhi order mein hon. Jaise, news classify karte waqt, *weather* aur *snow* jaise words *weather forecast* ka sanket de sakte hain, jabki *stock* aur *dollar* jaise words *financial news* ki taraf ishara karenge.

**Bag-of-Words** (BoW) vector representation sabse simple aur traditional vector representation hai. Har word ko ek vector index se jodte hain, aur ek vector element mein diye gaye document mein har word ke occurrences ki sankhya hoti hai.

![Yeh image dikhati hai ki bag-of-words vector representation memory mein kaise represent hota hai.](../../../../../lessons/5-NLP/13-TextRep/images/bag-of-words-example.png)

> **Note**: Aap BoW ko text ke individual words ke saare one-hot-encoded vectors ka sum bhi maan sakte hain.

Neeche Scikit Learn Python library use karke bag-of-words representation generate karne ka example diya gaya hai:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
sc_vectorizer = CountVectorizer()
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
sc_vectorizer.fit_transform(corpus)
sc_vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

Hum upar define kiye gaye Keras vectorizer ko bhi use kar sakte hain, har word number ko one-hot encoding mein convert karke aur un sabhi vectors ko add karke:

In [ ]:
def to_bow(text):
    return tf.reduce_sum(tf.one_hot(vectorizer(text),vocab_size),axis=0)

to_bow('My dog likes hot dogs on a hot day.').numpy()

> **Note**: Aapko yeh dekhkar hairaani ho sakti hai ki result pichle example se alag hai. Iska reason yeh hai ki Keras example mein vector ki length vocabulary size ke barabar hai, jo poore AG News dataset se bani thi, jabki Scikit Learn example mein humne sample text se hi vocabulary on the fly bana li.

## BoW classifier ko train karna

Ab jab humne apne text ka bag-of-words representation banana seekh liya hai, chaliye ek classifier train karte hain jo ise use kare. Sabse pehle, hamein apne dataset ko bag-of-words representation mein convert karna hoga. Yeh `map` function se is tarah kiya ja sakta hai:

In [ ]:
batch_size = 128

ds_train_bow = ds_train.map(lambda x: (to_bow(x['title']+x['description']),x['label'])).batch(batch_size)
ds_test_bow = ds_test.map(lambda x: (to_bow(x['title']+x['description']),x['label'])).batch(batch_size)

Ab chaliye ek simple classification neural network define karte hain jisme ek linear layer hai. Input size `vocab_size` hai, aur output size classes ki sankhya (4) ke barabar hai. Kyunki hum classification task solve kar rahe hain, final activation function **softmax** hai:

In [ ]:
model = keras.models.Sequential([
    keras.layers.Dense(4,activation='softmax',input_shape=(vocab_size,))
])
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
model.fit(ds_train_bow,validation_data=ds_test_bow)

Kyunki hamare paas 4 classes hain, 80% se zyada accuracy ek achha result hai.

## Classifier ko ek network ke roop mein train karna

Kyunki vectorizer bhi ek Keras layer hai, hum ek aisa network define kar sakte hain jisme yeh shamil ho, aur ise end-to-end train kar sakte hain. Is tarah hamein `map` se dataset ko vectorize karne ki zaroorat nahi padegi, hum bas original dataset ko network ke input mein pass kar sakte hain.

> **Note**: Phir bhi hamein apne dataset par `map` apply karna hoga taaki dictionary ke fields (jaise `title`, `description` aur `label`) ko tuples mein badal sakein. Lekin, jab disk se data load kar rahe hon, toh hum shuru mein hi required structure ke saath dataset bana sakte hain.

In [ ]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

inp = keras.Input(shape=(1,),dtype=tf.string)
x = vectorizer(inp)
x = tf.reduce_sum(tf.one_hot(x,vocab_size),axis=1)
out = keras.layers.Dense(4,activation='softmax')(x)
model = keras.models.Model(inp,out)
model.summary()

model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))


## Bigrams, Trigrams aur N-grams

Bag-of-words approach ki ek limitation yeh hai ki kuch words multi-word expressions ka hissa hote hain. Jaise, 'hot dog' ka meaning 'hot' aur 'dog' words se bilkul alag hota hai. Agar hum 'hot' aur 'dog' words ko hamesha ek hi vector se represent karte hain, toh yeh hamare model ko confuse kar sakta hai.

Is problem ko solve karne ke liye, **n-gram representations** ka use aksar document classification methods mein hota hai, jahan har word, bi-word ya tri-word ki frequency classification model ko train karne ke liye ek useful feature hoti hai. Jaise, bigram representation mein, hum original words ke alawa saare word pairs ko vocabulary mein add kar denge.

Neeche ek example hai jo dikhata hai ki Scikit Learn use karke bigram bag-of-words representation kaise banayi ja sakti hai:

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), token_pattern=r'\b\w+\b', min_df=1)
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
bigram_vectorizer.fit_transform(corpus)
print("Vocabulary:\n",bigram_vectorizer.vocabulary_)
bigram_vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()


n-gram approach ki main kamzori yeh hai ki vocabulary size bahut tezi se badhne lagta hai. Practice mein, hamein n-gram representation ko kisi dimensionality reduction technique, jaise *embeddings*, ke saath combine karna padta hai, jise hum agli unit mein discuss karenge.

Apne **AG News** dataset mein n-gram representation use karne ke liye, hamein `TextVectorization` constructor mein `ngrams` parameter pass karna hoga. Bigram vocabulary ki length **kaafi badi** hoti hai, hamare case mein yeh 1.3 million se zyada tokens hai! Isliye, bigram tokens ko bhi kisi reasonable number tak limit karna samajhdaari hogi.

Hum upar diye gaye code se classifier ko train kar sakte hain, lekin woh memory ke hisaab se bahut inefficient hoga. Agli unit mein, hum embeddings use karke bigram classifier train karenge. Tab tak, aap is notebook mein bigram classifier training ke saath experiment kar sakte hain aur dekh sakte hain ki kya aap higher accuracy paa sakte hain.

## BoW vectors ko automatically compute karna

Upar ke example mein, humne individual words ki one-hot encodings ko add karke BoW vector haath se compute kiya tha. Lekin, TensorFlow ke latest version mein, hum BoW vector ko automatically compute kar sakte hain, bas vectorizer constructor mein `output_mode='count` parameter pass karke. Isse hamare model ko define aur train karna kaafi aasaan ho jata hai:

In [ ]:
model = keras.models.Sequential([
    keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size,output_mode='count'),
    keras.layers.Dense(4,input_shape=(vocab_size,), activation='softmax')
])
print("Training vectorizer")
model.layers[0].adapt(ds_train.take(500).map(extract_text))
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

## Term Frequency - Inverse Document Frequency (TF-IDF)

BoW representation mein, words ke occurrences ko ek hi technique se weight kiya jata hai, chahe woh word koi bhi ho. Lekin, yeh clear hai ki *a* aur *in* jaise common words classification ke liye utne important nahi hote jitne specialized words. Zyadatar NLP tasks mein kuch words doosron se zyada relevant hote hain.

**TF-IDF** ka matlab hai **Term Frequency - Inverse Document Frequency**. Yeh bag-of-words ka ek variation hai, jisme document mein word ki presence dikhane wali binary 0/1 value ke bajaye, ek floating-point value use hoti hai, jo corpus mein word ke occurrence ki frequency se related hoti hai.

Formally, document $j$ mein word $i$ ka weight $w_{ij}$ is tarah define hota hai:
$$
w_{ij} = tf_{ij}\times\log({N\over df_i})
$$
jahan
* $tf_{ij}$ document $j$ mein $i$ ke occurrences ki sankhya hai, yaani woh BoW value jo humne pehle dekhi thi
* $N$ collection mein documents ki sankhya hai
* $df_i$ poori collection mein word $i$ ko contain karne wale documents ki sankhya hai

TF-IDF value $w_{ij}$ document mein word ke occurrences ki sankhya ke proportion mein badhti hai aur corpus mein un documents ki sankhya se offset hoti hai jinme woh word shamil hai. Yeh is baat ko adjust karne mein madad karta hai ki kuch words doosron se zyada baar dikhte hain. Jaise, agar koi word *har* document mein dikhta hai, toh $df_i=N$, aur $w_{ij}=0$, aur un words ko poori tarah ignore kar diya jayega.

Aap Scikit Learn use karke aasaani se text ka TF-IDF vectorization bana sakte hain:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

Keras mein, `TextVectorization` layer `output_mode='tf-idf'` parameter pass karke automatically TF-IDF frequencies compute kar sakti hai. Chaliye upar use kiya gaya code dohrate hain taaki dekh sakein ki TF-IDF use karne se accuracy badhti hai ya nahi:

In [ ]:
model = keras.models.Sequential([
    keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size,output_mode='tf-idf'),
    keras.layers.Dense(4,input_shape=(vocab_size,), activation='softmax')
])
print("Training vectorizer")
model.layers[0].adapt(ds_train.take(500).map(extract_text))
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

## Conclusion

Halanki TF-IDF representations alag-alag words ko frequency weights dete hain, woh na meaning express kar sakte hain na order. Jaisa ki mashhoor linguist J. R. Firth ne 1935 mein kaha tha, "word ka complete meaning hamesha contextual hota hai, aur context se alag meaning ki koi bhi study seriously nahi li ja sakti." Hum is course mein aage language modeling use karke text se contextual information capture karna seekhenge.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->